# Drug-Symptom-Relation Triplet Extraction
### BioBERT + spaCy Dependency Parsing on BC5CDR


In [ ]:
!pip -q install "transformers>=4.40,<5" "accelerate>=0.28" spacy scikit-learn
!python -m spacy download en_core_web_sm -q


In [ ]:
import re
from collections import deque

import pandas as pd
import spacy
import torch
from sklearn.model_selection import train_test_split
from spacy.matcher import PhraseMatcher
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.2'
LABELS = ['NO_RELATION', 'TREATS', 'CAUSES']
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
nlp = spacy.load('en_core_web_sm')
print('GPU available:', torch.cuda.is_available())

In [ ]:
import re
import random
from collections import Counter

# Download BC5CDR from official GitHub repo
!wget -q "https://github.com/JHnlp/BioCreative-V-CDR-Corpus/raw/master/CDR_Data.zip" -O CDR_Data.zip
!unzip -q CDR_Data.zip
!ls CDR_Data/CDR.Corpus.v010516/

def parse_pubtator(filepath):
    data = []
    with open(filepath) as f:
        content = f.read()
    blocks = content.strip().split("\n\n")
    for block in blocks:
        lines = block.strip().split("\n")
        if not lines:
            continue
        text = ""
        entities = {}
        relations = []
        for line in lines:
            parts = line.split("\t")
            if "|t|" in line:
                text += line.split("|t|")[1] + " "
            elif "|a|" in line:
                text += line.split("|a|")[1] + " "
            elif len(parts) == 6:
                _, start, end, mention, etype, eid = parts
                entities[eid] = (mention, etype)
            elif len(parts) == 4 and parts[1] == "CID":
                relations.append((parts[2], parts[3]))
        cid_pairs = set(relations)
        drugs    = {eid: m for eid, (m, t) in entities.items() if t == "Chemical"}
        diseases = {eid: m for eid, (m, t) in entities.items() if t == "Disease"}
        sentences = re.split(r"(?<=[.!?])\s+", text.strip())
        for d_id, drug in drugs.items():
            for s_id, disease in diseases.items():
                for sent in sentences:
                    if drug.lower() in sent.lower() and disease.lower() in sent.lower():
                        label = "CAUSES" if (d_id, s_id) in cid_pairs else "TREATS"
                        data.append((sent.strip(), drug, disease, label))
                        break
                else:
                    if random.random() < 0.15:
                        for sent in sentences:
                            if drug.lower() in sent.lower():
                                data.append((sent.strip(), drug, disease, "NO_RELATION"))
                                break
    return data

random.seed(42)
all_data = parse_pubtator("/content/CDR_Data/CDR.Corpus.v010516/CDR_TrainingSet.PubTator.txt")
# Balance classes to fix CAUSES F1 = 0 problem
causes = [r for r in all_data if r[3] == "CAUSES"]
treats = [r for r in all_data if r[3] == "TREATS"]
no_rel = [r for r in all_data if r[3] == "NO_RELATION"]

random.shuffle(treats)
treats = treats[:len(causes) * 2]  # cap TREATS at 2x CAUSES

balanced = causes + treats + no_rel
random.shuffle(balanced)

split = int(len(balanced) * 0.8)
training_data   = balanced[:split]
validation_data = balanced[split:]

print(f"Train : {len(training_data)}")
print(f"Val   : {len(validation_data)}")
print(f"Labels: {dict(Counter(r[3] for r in training_data))}")

DRUGS    = sorted({r[1].lower() for r in training_data})
SYMPTOMS = sorted({r[2].lower() for r in training_data})
print(f"Unique drugs: {len(DRUGS)}, Unique diseases: {len(SYMPTOMS)}")

for row in training_data[:2]:
    print(f"\n[{row[3]}] {row[1]} | {row[2]}")
    print(f"{row[0][:100]}...")

In [ ]:
def shortest_dependency_path(start, end):
    queue = deque([(start, [start])])
    visited = {start.i}
    while queue:
        token, path = queue.popleft()
        if token.i == end.i:
            return path
        neighbors = list(token.children)
        if token.head.i != token.i:
            neighbors.append(token.head)
        for neighbor in neighbors:
            if neighbor.i not in visited:
                visited.add(neighbor.i)
                queue.append((neighbor, path + [neighbor]))
    return []


def find_span(doc, phrase):
    start = doc.text.lower().find(phrase.lower())
    if start < 0:
        raise ValueError(f'{phrase!r} not found in {doc.text!r}')
    return doc.char_span(start, start + len(phrase), alignment_mode='expand')


def make_bert_input(sentence, drug, symptom):
    doc = nlp(sentence)
    drug_span = find_span(doc, drug)
    symptom_span = find_span(doc, symptom)
    path = shortest_dependency_path(drug_span.root, symptom_span.root)
    path_text = ' -> '.join(token.text for token in path)

    spans = [
        (drug_span.start_char, drug_span.end_char, '[DRUG]', '[/DRUG]'),
        (symptom_span.start_char, symptom_span.end_char, '[SYMPTOM]', '[/SYMPTOM]'),
    ]
    marked = sentence
    for start, end, opening, closing in sorted(spans, reverse=True):
        marked = marked[:start] + opening + marked[start:end] + closing + marked[end:]
    return f'{marked} [SEP] dependency path: {path_text}', path_text


class RelationDataset(Dataset):
    def __init__(self, rows, tokenizer):
        texts = [make_bert_input(s, d, y)[0] for s, d, y, _ in rows]
        self.encodings = tokenizer(
            texts, padding=True, truncation=True, max_length=256
        )
        self.labels = [LABEL2ID[label] for *_, label in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index])
        return item

## Fine-tune BioBERT on BC5CDR
Source: BioCreative V CDR corpus (NCBI) — 1,500 annotated PubMed abstracts.


In [ ]:
class RelationDataset(Dataset):
    def __init__(self, rows, tokenizer):
        texts = []
        self.labels = []
        for s, d, y, label in rows:
            try:
                text, _ = make_bert_input(s, d, y)
                texts.append(text)
                self.labels.append(LABEL2ID[label])
            except ValueError:
                # Skip rows where entity span not found in sentence
                continue
        self.encodings = tokenizer(
            texts, padding=True, truncation=True, max_length=256
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: torch.tensor(value[index]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index])
        return item

In [ ]:
train_rows      = training_data
validation_rows = validation_data

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({
    "additional_special_tokens": ["[DRUG]", "[/DRUG]", "[SYMPTOM]", "[/SYMPTOM]"]
})

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)
model.resize_token_embeddings(len(tokenizer))

args = TrainingArguments(
    output_dir="/content/triplet_model",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=RelationDataset(train_rows, tokenizer),
    eval_dataset=RelationDataset(validation_rows, tokenizer),
)

trainer.train()


## Extract Triplets


In [ ]:
drug_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
symptom_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

drug_matcher.add("DRUG", [nlp.make_doc(term) for term in DRUGS])
symptom_matcher.add("SYMPTOM", [nlp.make_doc(term) for term in SYMPTOMS])

TREAT_VERBS = {
    "relieve", "reduce", "improve", "treat",
    "control", "prescribe", "cure"
}

CAUSE_VERBS = {
    "cause", "produce", "trigger", "induce"
}


def detect_mentions(sentence):
    doc = nlp(sentence)
    drugs = [doc[start:end] for _, start, end in drug_matcher(doc)]
    symptoms = [doc[start:end] for _, start, end in symptom_matcher(doc)]
    return doc, drugs, symptoms


def rule_relation(path):
    lemmas = {token.lemma_.lower() for token in path}

    if lemmas & TREAT_VERBS:
        return "TREATS"

    if lemmas & CAUSE_VERBS:
        return "CAUSES"

    return None


def extract_triplets(sentences, threshold=0.45, max_dependency_distance=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    candidates = []

    for sentence in sentences:
        doc, drugs, symptoms = detect_mentions(sentence)

        for drug in drugs:
            for symptom in symptoms:
                path = shortest_dependency_path(drug.root, symptom.root)

                if path and len(path) - 1 <= max_dependency_distance:
                    bert_text, path_text = make_bert_input(
                        sentence, drug.text, symptom.text
                    )

                    candidates.append({
                        "sentence": sentence,
                        "drug": drug.text,
                        "symptom": symptom.text,
                        "dependency_path": path_text,
                        "rule_relation": rule_relation(path),
                        "bert_text": bert_text,
                    })

    columns = [
        "drug", "symptom", "relation", "bert_confidence",
        "method", "sentence", "dependency_path"
    ]

    if not candidates:
        return pd.DataFrame(columns=columns)

    batch = tokenizer(
        [row["bert_text"] for row in candidates],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        probabilities = torch.softmax(model(**batch).logits, dim=-1).cpu()

    results = []

    for row, scores in zip(candidates, probabilities):
        bert_confidence, label_id = scores.max(dim=-1)
        bert_confidence = float(bert_confidence)

        if row["rule_relation"] is not None:
            relation = row["rule_relation"]
            method = "Dependency rule"
        else:
            relation = ID2LABEL[int(label_id)]
            method = "BioBERT"

            if relation == "NO_RELATION" or bert_confidence < threshold:
                continue

        results.append({
            "drug": row["drug"],
            "symptom": row["symptom"],
            "relation": relation,
            "bert_confidence": round(bert_confidence, 4),
            "method": method,
            "sentence": row["sentence"],
            "dependency_path": row["dependency_path"],
        })

    return pd.DataFrame(results, columns=columns)


sentences = [
    "Aspirin relieved the patient's headache.",
    "Metformin caused nausea after two days.",
    "Ibuprofen reduced the patient's fever.",
    "Omeprazole improved her heartburn.",
    "Penicillin produced a severe rash.",
    "Paracetamol was prescribed for pain.",
    "Amoxicillin treated the bacterial infection.",
    "Ibuprofen caused vomiting.",
    "Insulin controlled high blood sugar.",
    "Amoxicillin caused diarrhea.",
]

results = extract_triplets(sentences, threshold=0.45)
display(results)

In [ ]:
from sklearn.metrics import classification_report, f1_score
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

val_texts, val_labels = [], []
for s, d, y, label in validation_data:
    try:
        text, _ = make_bert_input(s, d, y)
        val_texts.append(text)
        val_labels.append(LABEL2ID[label])
    except ValueError:
        continue

batch = tokenizer(
    val_texts, padding=True, truncation=True,
    max_length=256, return_tensors="pt"
).to(device)

with torch.no_grad():
    logits = model(**batch).logits
    preds = torch.argmax(logits, dim=-1).cpu().tolist()

# Get only labels that actually appear
present = sorted(set(val_labels + preds))
present_names = [LABELS[i] for i in present]

print(classification_report(
    val_labels, preds,
    labels=present,
    target_names=present_names,
    digits=3
))
print(f"Macro F1:    {f1_score(val_labels, preds, average='macro'):.3f}")
print(f"Weighted F1: {f1_score(val_labels, preds, average='weighted'):.3f}")